# 素因数分解の「地図」— 実験で「わかった」を接地させるノート

中学1年「素因数分解」のプロトタイプ（Python カーネル／JupyterLite でも動きます）

**このノートでやること**

1. 単元の **地図**（知識グラフ）を見る
2. **自分の地図** に、いま自分がどこまで納得しているかの **印** をつける
3. **実験** で確かめて、印を更新する
4. 地図の **成長** を見る

**印（接地レベル）は 5 段階**

| 印 | 意味 | 地図での見た目 |
|:--|:--|:--|
| 0 まだ | まだ触れていない | 点線の白 |
| 1 聞いた | 先生や教科書から聞いた | 白 |
| 2 たしかめた | 実験して、自分の目で見た | 薄い灰色 |
| 3 説明できる | なぜそうなるかを、自分の言葉で言える | 濃い灰色 |
| 4 証明できる | いつでも成り立つ理由を示せる | 黒 |

「たしかめた」と「証明できる」は別のもの。それがこのノートのいちばん大事なところです。

In [ ]:
# ============================================================
# 道具の準備（このセルの中身は読まなくて大丈夫です。実行だけしてください）
# ============================================================
import json, os, sys, copy, math, random
from datetime import datetime

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 太字のない同梱フォントで出る注意書きを出さない
from matplotlib.patches import FancyBboxPatch
import networkx as nx
import sympy

# ---------- 日本語フォント ----------
# 図の中の日本語を出すためのフォントを探す。見つける順番：
#   1. ノートと同じフォルダの fonts/ にある同梱フォント（Noto Sans CJK JP のサブセット）
#   2. パソコンに入っている日本語フォント
#   3. インターネットから取ってくる（1・2 がないときだけ。JupyterLite などで使う）
# どの場合も、その日本語フォントにない記号（≤ ⁴ ⚠ など）は DejaVu Sans で補う。
FONT_URLS = [
    "https://raw.githubusercontent.com/jxta/kg-grounding-proto/main/fonts/NotoSansCJKjp-Regular-subset.otf",  # 約 1.8 MB
    "https://jxta.github.io/kg-grounding-proto/files/fonts/NotoSansCJKjp-Regular-subset.otf",             # 同じもの（Pages）
    "https://raw.githubusercontent.com/notofonts/noto-cjk/main/Sans/OTF/Japanese/NotoSansCJKjp-Regular.otf",  # 約 16 MB
]
FONT_CANDIDATES = ["Noto Sans CJK JP", "Noto Sans JP", "Source Han Sans JP", "IPAexGothic", "IPAGothic",
                   "Hiragino Sans", "Hiragino Maru Gothic Pro", "Yu Gothic", "Meiryo", "MS Gothic",
                   "BIZ UDGothic", "TakaoGothic", "VL Gothic", "Noto Sans CJK SC"]

def _font_dirs():
    """fonts/ フォルダの候補（ノートのあるフォルダ、その上、ホームなど）"""
    dirs = ["fonts", os.path.join("..", "fonts"), os.path.expanduser("~/fonts"), os.path.expanduser("~/.fonts")]
    if sys.platform == "emscripten":                      # JupyterLite（Pyodide）：ドライブの浅い階層も見る
        dirs.append("/drive/fonts")
        try:
            for r, d, f in os.walk("/drive"):
                if r.count("/") <= 2:
                    dirs += [os.path.join(r, x) for x in d if x == "fonts"]
                else:
                    d[:] = []
        except Exception:
            pass
    return dirs

def _use_font_file(path):
    """フォントファイルを matplotlib に登録し、その名前を返す（読めなければ None）"""
    import shutil, tempfile
    import matplotlib.font_manager as fm
    try:
        tmp = os.path.join(tempfile.gettempdir(), os.path.basename(path))
        if os.path.abspath(path) != os.path.abspath(tmp):
            shutil.copyfile(path, tmp)                    # JupyterLite の仮想ドライブから読むより速くて確実
        fm.fontManager.addfont(tmp)
        return fm.FontProperties(fname=tmp).get_name()
    except Exception as e:
        print("※ フォントを読めませんでした:", path, e)
        return None

def _download_font():
    """同梱フォントも日本語フォントもないとき、ネットから取ってくる（失敗したら None）"""
    import tempfile
    for url in FONT_URLS:
        dst = os.path.join(tempfile.gettempdir(), url.rsplit("/", 1)[-1])
        try:
            if not os.path.exists(dst):
                if sys.platform == "emscripten":          # Pyodide：ワーカー内なら同期 XHR が使える
                    import js
                    xhr = js.XMLHttpRequest.new()
                    xhr.open("GET", url, False)
                    xhr.responseType = "arraybuffer"
                    xhr.send()
                    if xhr.status != 200:
                        continue
                    data = xhr.response.to_bytes()
                else:
                    import urllib.request
                    with urllib.request.urlopen(url, timeout=30) as r:
                        data = r.read()
                with open(dst, "wb") as f:
                    f.write(data)
            name = _use_font_file(dst)
            if name:
                print(f"※ 日本語フォントをダウンロードしました（{len(open(dst, 'rb').read()) // 1024} KB）")
                return name
        except Exception as e:
            print("※ ダウンロードできませんでした:", url.rsplit("/", 1)[-1], type(e).__name__)
    return None

def setup_font():
    import glob
    import matplotlib.font_manager as fm
    name = None
    for d in _font_dirs():                                # 1. 同梱フォント
        files = sorted(glob.glob(os.path.join(d, "*.otf")) + glob.glob(os.path.join(d, "*.ttf")))
        if files:
            name = _use_font_file(files[0])
            if name:
                break
    if name is None:                                      # 2. パソコンの日本語フォント
        installed = {f.name for f in fm.fontManager.ttflist}
        name = next((c for c in FONT_CANDIDATES if c in installed), None)
    if name is None:                                      # 3. ネットから
        name = _download_font()
    if name is None:
        try:
            import japanize_matplotlib  # noqa: F401
            name = "IPAexGothic"
        except Exception:
            return None
    # 日本語フォントにない記号は DejaVu Sans で補う（matplotlib 3.6 以降のフォールバック）
    matplotlib.rcParams["font.family"] = [name, "DejaVu Sans"]
    return name

FONT = setup_font()
matplotlib.rcParams.update({
    "axes.unicode_minus": False,
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#444444",
    "axes.labelcolor": "#222222",
    "xtick.color": "#444444",
    "ytick.color": "#444444",
    "axes.prop_cycle": matplotlib.cycler(color=["#000000", "#7a7a7a", "#bbbbbb"]),
})
if FONT is None:
    print("※ 日本語フォントが見つからないので、図の中の名前は英語IDで表示します。"
          "（ノートと同じフォルダに fonts/ を置くか、インターネットにつないで、このセルをもう一度実行してください）")

FIG_DIR = os.environ.get("KG_FIG_DIR")  # 図を PNG に保存したいときだけ設定
if FIG_DIR:
    os.makedirs(FIG_DIR, exist_ok=True)

def _save(fig, name):
    if FIG_DIR:
        fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight", facecolor="white")

def now():
    return datetime.now().strftime("%Y-%m-%d %H:%M")

In [ ]:
# ============================================================
# 単元の地図（教科書側の知識グラフ）— ノードは「事実」、矢印は「並べ方の一案」
# ============================================================
LEVELS = {0: "まだ", 1: "聞いた", 2: "たしかめた", 3: "説明できる", 4: "証明できる"}

UNIT = {
    "unit": "素因数分解（中学1年）",
    "nodes": {
        "divisor_multiple": {"label": "約数と倍数",              "kind": "定義", "pos": (5.5, 6),
            "hint": "12 の約数を全部書き出して、倍数との関係をたしかめる"},
        "composite":        {"label": "合成数",                  "kind": "定義", "pos": (2.5, 5),
            "hint": "篩で消された数はどんな数？（実験1）"},
        "prime":            {"label": "素数",                    "kind": "定義", "pos": (8.5, 5),
            "hint": "100 までの表で篩をかける（実験1）"},
        "factorization":    {"label": "素因数分解",              "kind": "性質", "pos": (4.0, 4),
            "hint": "大きな数を小さい素数で割っていく（実験2）"},
        "one_not_prime":    {"label": "1は素数ではない",         "kind": "約束", "pos": (7.0, 4),
            "hint": "1 を素数に入れると何が困る？（実験3）"},
        "sieve":            {"label": "エラトステネス\nの篩",     "kind": "方法", "pos": (11.0, 5),
            "hint": "10 までの素数で消すだけで 100 まで足りるのはなぜ？"},
        "exponent":         {"label": "累乗の表し方",            "kind": "書き方", "pos": (1.0, 3),
            "hint": "2×2×2×3×3 を短く書く（実験2）"},
        "trial_division":   {"label": "素因数分解の手順",        "kind": "方法", "pos": (4.8, 3),
            "hint": "割っていけば必ず終わるのはなぜ？（実験2）"},
        "uniqueness":       {"label": "分解はただ一通り\n（一意性）", "kind": "性質", "pos": (7.0, 3),
            "hint": "因数の木を違う順で作ってみる（実験3）"},
        "square":           {"label": "平方数",                  "kind": "性質", "pos": (1.0, 2),
            "hint": "平方数の素因数分解の指数を見る（実験6）"},
        "gcd_lcm":          {"label": "最大公約数・\n最小公倍数", "kind": "方法", "pos": (3.0, 2),
            "hint": "24 と 36 の素因数分解を並べて、共通の部分を見る"},
        "divisor_count":    {"label": "約数の個数",              "kind": "性質", "pos": (7.0, 2),
            "hint": "約数の個数と指数の関係を表にする（実験5）"},
        "infinite_primes":  {"label": "素数は無限にある",        "kind": "性質", "pos": (9.8, 3),
            "hint": "小さい素数を全部かけて 1 を足す（実験7）", "advanced": True},
        "prime_race":       {"label": "素数の偏り\n（4で割った余り）", "kind": "性質", "pos": (11.8, 3),
            "hint": "4 で割って 1 余る素数と 3 余る素数を数える（実験8）", "advanced": True},
    },
    # 矢印は「教科書ならこう並べる」という一案。きみの地図では別の道があってよい。
    "edges": [
        ("divisor_multiple", "prime",           "前提"),
        ("divisor_multiple", "composite",       "前提"),
        ("prime",            "one_not_prime",   "約束"),
        ("prime",            "sieve",           "方法"),
        ("prime",            "factorization",   "前提"),
        ("composite",        "factorization",   "前提"),
        ("factorization",    "trial_division",  "方法"),
        ("factorization",    "exponent",        "書き方"),
        ("factorization",    "uniqueness",      "性質"),
        ("one_not_prime",    "uniqueness",      "理由"),
        ("uniqueness",       "divisor_count",   "使う"),
        ("factorization",    "gcd_lcm",         "使う"),
        ("exponent",         "square",          "使う"),
        ("prime",            "infinite_primes", "発展"),
        ("prime",            "prime_race",      "発展"),
    ],
}

REF = nx.DiGraph()
for k, v in UNIT["nodes"].items():
    REF.add_node(k, **v)
for a, b, kind in UNIT["edges"]:
    REF.add_edge(a, b, kind=kind)

def label_of(node):
    return UNIT["nodes"][node]["label"] if FONT else node

def name_of(node):
    return UNIT["nodes"][node]["label"].replace("\n", "")

with open("unit_prime_factorization.json", "w", encoding="utf-8") as f:
    json.dump(UNIT, f, ensure_ascii=False, indent=1)
print(f"単元：{UNIT['unit']}　ノード {REF.number_of_nodes()} 個、矢印 {REF.number_of_edges()} 本")

In [ ]:
# ============================================================
# 地図を描く道具（モノクロ）
# ============================================================
BOX_W, BOX_H = 1.5, 0.5
XLIM, YLIM = (-0.4, 12.9), (1.05, 6.6)

# 接地レベルの見た目：白 → 薄い灰 → 濃い灰 → 黒
STYLE = {
    0: dict(fc="white",   ec="#888888", lw=0.8, ls=(0, (2, 2)), tc="#777777"),
    1: dict(fc="white",   ec="black",   lw=0.9, ls="-",         tc="black"),
    2: dict(fc="#d9d9d9", ec="black",   lw=0.9, ls="-",         tc="black"),
    3: dict(fc="#7a7a7a", ec="black",   lw=0.9, ls="-",         tc="white"),
    4: dict(fc="black",   ec="black",   lw=0.9, ls="-",         tc="white"),
}

def _boundary(c, d, w=BOX_W, h=BOX_H):
    """箱（中心 c、幅 w、高さ h）の中心から方向 d に進んで箱のふちに出る点"""
    dx, dy = d
    tx = (w / 2) / abs(dx) if dx else float("inf")
    ty = (h / 2) / abs(dy) if dy else float("inf")
    t = min(tx, ty)
    return (c[0] + dx * t, c[1] + dy * t)

def _bezier(s, e, rad, n=40):
    """arc3 と同じ曲線（2 次ベジェ）の点列。rad>0 で進行方向の右へふくらむ"""
    dx, dy = e[0] - s[0], e[1] - s[1]
    cx, cy = (s[0] + e[0]) / 2 + rad * dy, (s[1] + e[1]) / 2 - rad * dx
    pts = []
    for i in range(1, n):
        t = i / n
        pts.append(((1 - t) ** 2 * s[0] + 2 * (1 - t) * t * cx + t ** 2 * e[0],
                    (1 - t) ** 2 * s[1] + 2 * (1 - t) * t * cy + t ** 2 * e[1]))
    return pts

def _hits(pts, boxes, w=BOX_W, h=BOX_H, margin=0.05):
    return sum(any(abs(x - c[0]) < w / 2 + margin and abs(y - c[1]) < h / 2 + margin for x, y in pts) for c in boxes)

def draw_arrow(ax, p, q, color="#9a9a9a", lw=1.0, ls="-", gap=0.06, zorder=1, scale=1.0, avoid=(), offset=0.0):
    """箱 p から箱 q へ矢印。avoid の箱を横切るときは、横切らない側へ曲げる。offset は線と直交する向きのずらし"""
    dx, dy = q[0] - p[0], q[1] - p[1]
    L = math.hypot(dx, dy)
    ux, uy = dx / L, dy / L
    if offset:
        p = (p[0] + uy * offset, p[1] - ux * offset)
        q = (q[0] + uy * offset, q[1] - ux * offset)
    s = _boundary(p, (ux, uy)); e = _boundary(q, (-ux, -uy))
    s = (s[0] + ux * gap, s[1] + uy * gap); e = (e[0] - ux * gap, e[1] - uy * gap)
    rad = 0.0
    if avoid:
        best = None
        for r in (0.0, 0.22, -0.22, 0.32, -0.32, 0.45, -0.45):
            n_hit = _hits(_bezier(s, e, r), avoid)
            if best is None or n_hit < best[0]:
                best = (n_hit, r)
            if n_hit == 0:
                break
        rad = best[1]
    ax.annotate("", xy=e, xytext=s, zorder=zorder,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, linestyle=ls,
                                shrinkA=0, shrinkB=0, mutation_scale=11 * scale,
                                connectionstyle=f"arc3,rad={rad}"))

def draw_node(ax, c, text, style, fontsize=9, w=BOX_W, h=BOX_H, zorder=3, halo=False, mark=None):
    box = FancyBboxPatch((c[0] - w / 2, c[1] - h / 2), w, h,
                         boxstyle="round,pad=0,rounding_size=0.12",
                         fc=style["fc"], ec=style["ec"], lw=style["lw"], ls=style["ls"], zorder=zorder)
    ax.add_patch(box)
    if halo:  # 変化した箱：外側にもう一つ枠を描く（白でも黒でも見える）
        ax.add_patch(FancyBboxPatch((c[0] - w / 2 - 0.09, c[1] - h / 2 - 0.09), w + 0.18, h + 0.18,
                                    boxstyle="round,pad=0,rounding_size=0.18", fc="none", ec="black",
                                    lw=1.3, zorder=zorder - 1))
    ax.text(c[0], c[1], text, ha="center", va="center", fontsize=fontsize, color=style["tc"],
            zorder=zorder + 1, linespacing=1.15)
    if mark:  # 右上に小さな印（例：↑）
        ax.text(c[0] + w / 2 - 0.05, c[1] + h / 2 - 0.02, mark, ha="right", va="top",
                fontsize=fontsize * 0.9, color=style["tc"], zorder=zorder + 2, fontweight="bold")

def _canvas(ax):
    ax.set_xlim(*XLIM); ax.set_ylim(*YLIM)
    ax.set_aspect("equal"); ax.axis("off")

def _fontsize(ax):
    fig = ax.get_figure()
    bbox = ax.get_position()
    w_in = fig.get_size_inches()[0] * bbox.width
    unit = w_in / (XLIM[1] - XLIM[0])
    return max(5.0, min(9.0, 9.0 * unit / 0.9))

def draw_legend(ax, fontsize=8, y=1.32, x0=-0.2, step=2.6):
    for i, lv in enumerate(range(5)):
        x = x0 + i * step
        st = STYLE[lv]
        box = FancyBboxPatch((x, y - 0.11), 0.42, 0.22, boxstyle="round,pad=0,rounding_size=0.05",
                             fc=st["fc"], ec=st["ec"], lw=st["lw"], ls=st["ls"], zorder=3)
        ax.add_patch(box)
        ax.text(x + 0.55, y, f"{lv}  {LEVELS[lv]}", ha="left", va="center", fontsize=fontsize, color="#333333")

def draw_reference(ax=None, edge_labels=False, title=None, fontsize=None, legend=False):
    """教科書側の地図（全ノード白）"""
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(12, 5.3))
    _canvas(ax)
    fs = fontsize or _fontsize(ax)
    for a, b, d in REF.edges(data=True):
        pa, pb = REF.nodes[a]["pos"], REF.nodes[b]["pos"]
        draw_arrow(ax, pa, pb, color="#555555", lw=1.0)
        if edge_labels:
            mx, my = (pa[0] + pb[0]) / 2, (pa[1] + pb[1]) / 2
            ax.text(mx, my, d["kind"], fontsize=fs * 0.75, color="#555555", ha="center", va="center",
                    bbox=dict(boxstyle="square,pad=0.15", fc="white", ec="none"), zorder=2)
    for k, v in REF.nodes(data=True):
        st = dict(STYLE[1]); 
        if v.get("advanced"):
            st = dict(STYLE[1], ls=(0, (3, 2)))
        draw_node(ax, v["pos"], label_of(k), st, fontsize=fs)
    if legend:
        draw_legend(ax, fontsize=fs * 0.9)
    ax.set_title(title or f"単元の地図：{UNIT['unit']}（点線＝発展）", fontsize=fs + 2, loc="left", color="#222222")
    if own:
        _save(ax.get_figure(), "ref_map")
        plt.show()
    return ax

In [ ]:
# ============================================================
# 自分の地図の道具：印をつける（mark）、道をつなぐ（link）、記録する（snapshot）、
# 描く（show_map）、成長を見る（diff / show_history）、問いかけ（ask_me）
# ============================================================
import unicodedata
def _w(t):
    return sum(2 if unicodedata.east_asian_width(ch) in "WF" else 1 for ch in t)

def _pad(t, n):
    return t + " " * max(0, n - _w(t))

def new_map(name="わたし"):
    return {"name": name, "unit": UNIT["unit"], "created": now(),
            "nodes": {k: {"level": 0, "why": "", "evidence": [], "uses": []} for k in UNIT["nodes"]},
            "edges": [], "log": [], "snapshots": []}

def heard(m, nodes=None):
    """授業で聞いたところに「聞いた」の印をつける（発展以外はすべて、が既定）"""
    nodes = nodes or [k for k, v in UNIT["nodes"].items() if not v.get("advanced")]
    for k in nodes:
        if m["nodes"][k]["level"] < 1:
            m["nodes"][k]["level"] = 1
    m["log"].append({"t": now(), "what": "heard", "nodes": nodes})
    print(f"「聞いた」の印：{len(nodes)} 個")

def mark(m, node, level, why="", evidence=None, uses=None):
    """ノードに接地レベルの印をつける。why は自分の言葉で、evidence はどの実験か。
    uses は「この説明で使った知識」（ほかのノード）。"""
    assert node in m["nodes"], f"知らないノード: {node}"
    assert level in LEVELS, "level は 0〜4"
    n = m["nodes"][node]
    old = n["level"]
    n["level"] = level
    if why:
        n["why"] = why
    if evidence:
        ev = evidence if isinstance(evidence, list) else [evidence]
        n["evidence"] = list(dict.fromkeys(n["evidence"] + ev))
    if uses:
        n["uses"] = list(dict.fromkeys(n["uses"] + list(uses)))
    m["log"].append({"t": now(), "what": "mark", "node": node, "from": old, "to": level,
                     "why": why, "evidence": evidence, "uses": uses})
    arrow = "↑" if level > old else ("↓" if level < old else "＝")
    ev = f"｜根拠：{', '.join(n['evidence'])}" if n["evidence"] else ""
    print(f"{arrow} {name_of(node)}：{LEVELS[old]} → {LEVELS[level]}{ev}")
    if why:
        print(f"   わけ：{why}")
    for u in (uses or []):
        lu = m["nodes"][u]["level"]
        if lu < level:
            print(f"   ⚠ この説明は「{name_of(u)}」を使っているが、それは『{LEVELS[lu]}』止まり")

def link(m, a, b, why=""):
    """自分の道（矢印）をつなぐ"""
    assert a in m["nodes"] and b in m["nodes"]
    for e in m["edges"]:
        if e["from"] == a and e["to"] == b:
            e["why"] = why or e["why"]; return
    m["edges"].append({"from": a, "to": b, "why": why, "t": now()})
    m["log"].append({"t": now(), "what": "link", "from": a, "to": b, "why": why})
    print(f"＋ 道：{name_of(a)} → {name_of(b)}" + (f"（{why}）" if why else ""))

def snapshot(m, label):
    s = {"label": label, "t": now(),
         "levels": {k: v["level"] for k, v in m["nodes"].items()},
         "edges": [(e["from"], e["to"]) for e in m["edges"]]}
    m["snapshots"].append(s)
    print(f"記録しました：[{len(m['snapshots'])-1}] {label}（{s['t']}）")
    return len(m["snapshots"]) - 1

def weak_uses(m):
    """『説明できる/証明できる』の説明が、より弱いノードを使っている箇所"""
    out = []
    for k, v in m["nodes"].items():
        for u in v["uses"]:
            if m["nodes"][u]["level"] < v["level"]:
                out.append((u, k))
    return out

def check(m):
    ws = weak_uses(m)
    if not ws:
        print("説明の材料はすべて、それ以上の強さで接地しています。")
    for u, k in ws:
        print(f"⚠ 「{name_of(k)}」（{LEVELS[m['nodes'][k]['level']]}）の説明は"
              f"「{name_of(u)}」（{LEVELS[m['nodes'][u]['level']]}）を使っている\n"
              f"   → 「{name_of(u)}」が正しい『とすれば』、の話になっている。")
    return ws

def show_map(m, title=None, compare=None, ref_edges=True, ax=None, fontsize=None, legend=True, save=None, labels=True):
    """自分の地図を描く。compare=記録番号 を渡すと、その記録からの変化を二重枠と↑で示す。labels=False で名前なしの縮小版"""
    own = ax is None
    if own:
        fig, ax = plt.subplots(figsize=(12, 5.3))
    _canvas(ax)
    fs = fontsize or _fontsize(ax)
    base = m["snapshots"][compare] if compare is not None else None
    P = {k: v["pos"] for k, v in UNIT["nodes"].items()}
    def others(a, b):
        return [P[k] for k in P if k not in (a, b)]
    mine = {(e["from"], e["to"]) for e in m["edges"]}
    weak = weak_uses(m)
    if ref_edges:  # 教科書の道（薄く）
        for a, b in REF.edges():
            draw_arrow(ax, P[a], P[b], color="#c8c8c8", lw=0.8, zorder=1, avoid=others(a, b))
    for e in m["edges"]:  # 自分の道（黒）。逆向きの道があるときは少しずらす
        a, b = e["from"], e["to"]
        new = base is not None and (a, b) not in base["edges"]
        off = 0.09 if (REF.has_edge(b, a) or (b, a) in mine) else 0.0
        draw_arrow(ax, P[a], P[b], color="black", lw=2.0 if new else 1.4, zorder=2, avoid=others(a, b), offset=off)
    for u, k in weak:  # 弱い材料を使っている説明（黒の破線）。自分の道と重なるときは横にずらす
        off = -0.11 if (u, k) in mine else 0.0
        draw_arrow(ax, P[u], P[k], color="black", lw=1.6, ls=(0, (4, 2)), zorder=2, avoid=others(u, k), offset=off)
    for k, v in UNIT["nodes"].items():
        lv = m["nodes"][k]["level"]
        changed = base is not None and lv != base["levels"][k]
        mk = None
        if changed:
            mk = "↑" if lv > base["levels"][k] else "↓"
        draw_node(ax, v["pos"], label_of(k) if labels else "", STYLE[lv], fontsize=fs, halo=changed, mark=mk if labels else None)
    if legend:
        draw_legend(ax, fontsize=fs * 0.9)
    t = title or f"{m['name']}の地図（{now()}）"
    if base is not None:
        t += f"　二重枠＝「{base['label']}」からの変化"
    ax.set_title(t, fontsize=fs + 2, loc="left", color="#222222")
    if own:
        _save(ax.get_figure(), save or "my_map")
        plt.show()
    return ax

def diff(m, i, j):
    a, b = m["snapshots"][i], m["snapshots"][j]
    print(f"[{a['label']}] → [{b['label']}]")
    ups = [(k, a["levels"][k], b["levels"][k]) for k in a["levels"] if a["levels"][k] != b["levels"][k]]
    for k, x, y in sorted(ups, key=lambda t: -(t[2] - t[1])):
        arrow = "↑" if y > x else "↓"
        print(f"  {arrow} {_pad(name_of(k), 26)}{LEVELS[x]} → {LEVELS[y]}")
    for e in m["edges"]:
        if (e["from"], e["to"]) in b["edges"] and (e["from"], e["to"]) not in a["edges"]:
            print(f"  ＋ 道：{name_of(e['from'])} → {name_of(e['to'])}" + (f"（{e['why']}）" if e["why"] else ""))
    if not ups:
        print("  （印の変化なし）")

def show_history(m, cols=None, save="history", labels=False):
    """記録した地図を並べて、成長をひと目で見る（既定は名前なしの縮小版。位置は本物の地図と同じ）"""
    n = len(m["snapshots"])
    cols = cols or n
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=((6.3 if labels else 4.4) * cols, (2.9 if labels else 2.1) * rows), squeeze=False)
    cur_edges = [e for e in m["edges"]]
    for idx, s in enumerate(m["snapshots"]):
        ax = axes[idx // cols][idx % cols]
        tmp = copy.deepcopy(m)   # 記録時点の地図を一時的に復元して描く
        for k in tmp["nodes"]:
            tmp["nodes"][k]["level"] = s["levels"][k]
        tmp["edges"] = [e for e in cur_edges if (e["from"], e["to"]) in s["edges"]]
        show_map(tmp, title=f"[{idx}] {s['label']}", ax=ax, legend=False, labels=labels)
    for idx in range(n, rows * cols):
        axes[idx // cols][idx % cols].axis("off")
    fig.tight_layout()
    _save(fig, save)
    plt.show()

def ask_me(m, k=6):
    """地図を見て問いかける（答えは言わない・地図は作らない）。問いの種類ごとに順番に出す"""
    cats = {"weak": [], "predict": [], "claim2": [], "heard": [], "method2": [], "edge": [], "def2": [], "explain3": [], "connect": []}
    for u, node in weak_uses(m):
        cats["weak"].append(f"「{name_of(node)}」の説明は「{name_of(u)}」を使っている。でも「{name_of(u)}」は"
                            f"『{LEVELS[m['nodes'][u]['level']]}』止まり。この説明は何を前提にしている？ それでいい？")
    for node, v in m["nodes"].items():
        info = UNIT["nodes"][node]; lv = v["level"]; kind = info["kind"]; h = info["hint"]
        if lv <= 1 and not info.get("advanced"):
            cats["heard"].append(f"「{name_of(node)}」はまだ『{LEVELS[lv]}』。実験で確かめられる？ ヒント：{h}")
        elif lv == 2 and info.get("advanced"):
            cats["predict"].append(f"「{name_of(node)}」はどこまで見た？ この先はどうなると思う？ 予想を書いてみよう（当たっても外れても記録する）")
        elif lv == 2 and kind == "性質":
            cats["claim2"].append(f"「{name_of(node)}」はどこまで確かめた？ その先でも成り立つと言える？ 『いつでも』と言うには何が要る？")
        elif lv == 2 and kind == "方法":
            cats["method2"].append(f"「{name_of(node)}」はうまくいった。なぜその方法でうまくいく？ ヒント：{h}")
        elif lv == 2:
            cats["def2"].append(f"「{name_of(node)}」を自分の言葉で言うと？ 例を 2 つと、例でないものを 1 つ挙げられる？")
        elif lv == 3:
            cats["explain3"].append(f"「{name_of(node)}」を、一度も習っていない友だちに説明するとしたら、最初の一文は？")
    for e in m["edges"]:
        if not e["why"]:
            cats["edge"].append(f"きみの道「{name_of(e['from'])} → {name_of(e['to'])}」：どんなつながり？ ひとことで。")
    mine = {(e["from"], e["to"]) for e in m["edges"]}
    for a, b in REF.edges():
        if (a, b) not in mine and m["nodes"][a]["level"] >= 2 and m["nodes"][b]["level"] >= 2:
            cats["connect"].append(f"「{name_of(a)}」と「{name_of(b)}」はどちらも確かめた。この二つはつながっている？ つながっているなら、どう？")
    qs, i = [], 0
    while len(qs) < k and any(cats.values()):   # 種類を順ぐりに
        for c in cats.values():
            if c and len(qs) < k:
                qs.append(c.pop(0))
    for i, q in enumerate(qs, 1):
        print(f"Q{i}. {q}")
    return qs

def save_map(m, path="my_map.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(m, f, ensure_ascii=False, indent=1)
    print(f"保存しました：{path}")

def load_map(path="my_map.json"):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

In [ ]:
# ============================================================
# 実験の道具（篩の表、因数の木、約数の表、素数レース）
# ============================================================
def sieve(n):
    """エラトステネスの篩。各数を「消した素数」を返す（素数は 0、1 は -1）"""
    crossed = [0] * (n + 1)
    crossed[1] = -1
    for p in range(2, n + 1):
        if crossed[p] == 0:
            for q in range(p * p, n + 1, p):
                if crossed[q] == 0:
                    crossed[q] = p
    return crossed

def sieve_grid(n=100, upto=None, title=None, save="sieve_grid"):
    """1〜n の表。消された数は薄く、消した素数を右下に小さく書く。upto を指定すると、その素数まで消した途中経過"""
    crossed = sieve(n)
    if upto is not None:
        crossed = [c if (c in (0, -1) or c <= upto) else 0 for c in crossed]
    cols = 10
    rows = math.ceil(n / cols)
    fig, ax = plt.subplots(figsize=(7.2, 0.72 * rows + 0.6))
    ax.set_xlim(0, cols); ax.set_ylim(0, rows); ax.set_aspect("equal"); ax.axis("off")
    for k in range(1, n + 1):
        r, c = divmod(k - 1, cols)
        x, y = c, rows - 1 - r
        st = crossed[k]
        if st == -1:      # 1
            ax.add_patch(FancyBboxPatch((x + 0.06, y + 0.06), 0.88, 0.88, boxstyle="square,pad=0",
                                        fc="white", ec="#666666", lw=0.9, ls=(0, (2, 2))))
            ax.text(x + 0.5, y + 0.5, "1", ha="center", va="center", fontsize=12, color="#444444")
            ax.text(x + 0.9, y + 0.12, "?", ha="right", va="bottom", fontsize=8, color="#444444")
        elif st == 0:     # 素数
            ax.add_patch(FancyBboxPatch((x + 0.06, y + 0.06), 0.88, 0.88, boxstyle="square,pad=0",
                                        fc="white", ec="black", lw=1.2))
            ax.text(x + 0.5, y + 0.5, str(k), ha="center", va="center", fontsize=12, color="black", fontweight="bold")
        else:             # 消された（合成数）
            ax.add_patch(FancyBboxPatch((x + 0.06, y + 0.06), 0.88, 0.88, boxstyle="square,pad=0",
                                        fc="#e6e6e6", ec="#bbbbbb", lw=0.6))
            ax.text(x + 0.5, y + 0.55, str(k), ha="center", va="center", fontsize=11, color="#9a9a9a")
            ax.text(x + 0.9, y + 0.12, str(st), ha="right", va="bottom", fontsize=6.5, color="#666666")
    t = title or (f"1〜{n} の表：太枠＝残った数（素数）、薄い数＝消された数（右下は消した素数）"
                  if upto is None else f"1〜{n} の表：{upto} までの素数で消したところ")
    ax.set_title(t, fontsize=9.5, loc="left", color="#222222")
    _save(fig, save)
    plt.show()
    return [k for k in range(2, n + 1) if crossed[k] == 0]

# ---------- 因数の木 ----------
def factor_tree(n, is_prime=None, splits=None, rng=None):
    """n を二つの因数に分けていく木。splits={n:(a,b)} で分け方を指定、なければ最小の素因数で分ける（rng で無作為）。
    戻り値：('leaf', n) または ('node', n, 左, 右)"""
    is_prime = is_prime or sympy.isprime
    if is_prime(n):
        return ("leaf", n)
    if splits and n in splits:
        a, b = splits[n]
    elif rng is not None:
        divs = [d for d in range(2, n) if n % d == 0 and d * d <= n]
        a = rng.choice(divs); b = n // a
    else:
        a = min(sympy.factorint(n)); b = n // a
    return ("node", n, factor_tree(a, is_prime, splits, rng), factor_tree(b, is_prime, splits, rng))

def leaves(t):
    return [t[1]] if t[0] == "leaf" else leaves(t[2]) + leaves(t[3])

def _depth(t):
    return 1 if t[0] == "leaf" else 1 + max(_depth(t[2]), _depth(t[3]))

def _layout(t, depth=0, x0=0, acc=None):
    """葉を左から順に並べ、親は子の真ん中に置く（戻り値：{id(node): (x, y)} と次の x）"""
    acc = acc if acc is not None else {}
    if t[0] == "leaf":
        acc[id(t)] = (x0, -depth)
        return acc, x0 + 1
    acc, x1 = _layout(t[2], depth + 1, x0, acc)
    acc, x2 = _layout(t[3], depth + 1, x1, acc)
    acc[id(t)] = ((acc[id(t[2])][0] + acc[id(t[3])][0]) / 2, -depth)
    return acc, x2

def _draw_tree(ax, t, pos, fs=10):
    x, y = pos[id(t)]
    if t[0] == "leaf":
        ax.add_patch(FancyBboxPatch((x - 0.32, y - 0.22), 0.64, 0.44, boxstyle="round,pad=0,rounding_size=0.1",
                                    fc="black", ec="black"))
        ax.text(x, y, str(t[1]), ha="center", va="center", fontsize=fs, color="white", fontweight="bold")
        return
    ax.add_patch(FancyBboxPatch((x - 0.36, y - 0.22), 0.72, 0.44, boxstyle="round,pad=0,rounding_size=0.1",
                                fc="white", ec="black", lw=0.9))
    ax.text(x, y, str(t[1]), ha="center", va="center", fontsize=fs, color="black")
    for child in (t[2], t[3]):
        cx, cy = pos[id(child)]
        ax.plot([x, cx], [y - 0.22, cy + 0.22], color="#666666", lw=0.9, zorder=0)
        _draw_tree(ax, child, pos, fs)

def show_trees(trees, titles=None, save="factor_trees", note=True):
    """いくつかの因数の木を並べて描く"""
    k = len(trees)
    depth = max(_depth(t) for t in trees)
    width = max(len(leaves(t)) for t in trees)
    fig, axes = plt.subplots(1, k, figsize=(0.85 * width * k + 0.8, 1.0 * depth + 1.2), squeeze=False)
    for i, t in enumerate(trees):
        ax = axes[0][i]
        pos, _ = _layout(t)
        n_leaf = len(leaves(t))
        xs = [p[0] for p in pos.values()]
        cx = (min(xs) + max(xs)) / 2
        pos = {kk: (p[0] - cx, p[1]) for kk, p in pos.items()}
        ax.set_xlim(-width / 2 - 0.6, width / 2 + 0.6); ax.set_ylim(-(depth - 1) - 1.0, 0.6)
        ax.set_aspect("equal"); ax.axis("off")
        _draw_tree(ax, t, pos)
        lv = sorted(leaves(t))
        if note:
            ax.text(0, -(depth - 1) - 0.75, "葉をそろえると： " + " × ".join(map(str, lv)),
                    ha="center", va="center", fontsize=9.5, color="#222222")
        if titles:
            ax.set_title(titles[i], fontsize=10, loc="left", color="#222222")
    fig.tight_layout()
    _save(fig, save)
    plt.show()

def sup(e):
    """指数を上付き文字に（2 → ²）"""
    return str(e).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))

def factorization_str(n):
    """素因数分解を 2³ × 3² の形の文字列で"""
    if n == 1:
        return "1"
    f = sympy.factorint(n)
    return " × ".join(f"{p}{sup(e)}" if e > 1 else f"{p}" for p, e in sorted(f.items()))

# ---------- 約数の表 ----------
def divisor_grid(n, save="divisor_grid"):
    """n の約数を「素因数ごとの指数の組み合わせ」の表として描く（素因数 2 種類まで）"""
    f = sorted(sympy.factorint(n).items())
    assert 1 <= len(f) <= 2, "素因数が 1〜2 種類の数で"
    (p, e), = f[:1]
    q, g = f[1] if len(f) == 2 else (None, 0)
    rows = [p ** i for i in range(e + 1)]
    cols = [q ** j for j in range(g + 1)] if q else [1]
    fig, ax = plt.subplots(figsize=(1.1 * (len(cols) + 1) + 1.2, 0.6 * (len(rows) + 1) + 0.9))
    ax.set_xlim(-0.2, len(cols) + 1); ax.set_ylim(-0.2, len(rows) + 1); ax.axis("off"); ax.set_aspect("equal")
    H = len(rows) + 1
    for j, cv in enumerate(cols):
        lab = f"{q}{sup(j)}" if q and j > 1 else (str(cv))
        ax.text(j + 1.5, H - 0.5, lab, ha="center", va="center", fontsize=10, color="#444444")
    for i, rv in enumerate(rows):
        lab = f"{p}{sup(i)}" if i > 1 else str(rv)
        ax.text(0.5, H - 1.5 - i, lab, ha="center", va="center", fontsize=10, color="#444444")
        for j, cv in enumerate(cols):
            ax.add_patch(FancyBboxPatch((j + 1.05, H - 1.95 - i), 0.9, 0.9, boxstyle="square,pad=0",
                                        fc="#eeeeee", ec="#999999", lw=0.6))
            ax.text(j + 1.5, H - 1.5 - i, str(rv * cv), ha="center", va="center", fontsize=11, color="black")
    ax.plot([1, 1], [0, H - 1], color="#444444", lw=0.8); ax.plot([0, len(cols) + 1], [H - 1, H - 1], color="#444444", lw=0.8)
    n_div = (e + 1) * (g + 1)
    ax.set_title(f"{n} = {factorization_str(n)} の約数： {e+1} 行 × {g+1} 列 = {n_div} 個", fontsize=10, loc="left", color="#222222")
    _save(fig, save)
    plt.show()

# ---------- 素数レース ----------
def prime_race(N=30000, save="prime_race"):
    """4 で割って 3 余る素数の個数 − 1 余る素数の個数、を x まで数えて描く"""
    s = np.ones(N + 1, dtype=bool); s[:2] = False
    for i in range(2, int(N ** 0.5) + 1):
        if s[i]:
            s[i * i::i] = False
    primes = np.nonzero(s)[0]
    d = np.zeros(N + 1, dtype=int)
    d[primes[primes % 4 == 3]] += 1
    d[primes[primes % 4 == 1]] -= 1
    D = np.cumsum(d)
    x = np.arange(N + 1)
    neg = np.nonzero(D < 0)[0]
    first = int(neg[0]) if len(neg) else None
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={"width_ratios": [2.2, 1]})
    ax1.step(x, D, where="post", color="black", lw=0.8)
    ax1.axhline(0, color="#999999", lw=0.8)
    ax1.set_xlabel("x"); ax1.set_ylabel("（3余る素数）−（1余る素数）")
    ax1.set_title(f"x 以下の素数を数える：4 で割って 3 余る方が多い？（x は {N} まで）", fontsize=10, loc="left", color="#222222")
    ax1.set_ylim(min(D.min(), 0) - 0.3 * D.max(), D.max() * 1.1)
    if first:
        ax1.annotate(f"x = {first}：初めて 1 余る方が多くなる", xy=(first, D[first]), xytext=(first * 0.35, -0.22 * D.max()),
                     fontsize=9, color="#222222", va="center",
                     arrowprops=dict(arrowstyle="-|>", color="#222222", lw=0.8, shrinkB=3))
        lo, hi = first - 300, first + 300
        ax2.step(x[lo:hi], D[lo:hi], where="post", color="black", lw=1.0)
        ax2.plot([first], [D[first]], "o", color="black", ms=4)
        ax2.axhline(0, color="#999999", lw=0.8)
        ax2.set_xlim(lo, hi); ax2.set_title(f"{first} のまわりを拡大", fontsize=10, loc="left", color="#222222")
        ax2.set_xlabel("x")
    fig.tight_layout()
    _save(fig, save)
    plt.show()
    lead3 = int(np.sum(D[2:] > 0)); lead1 = int(np.sum(D[2:] < 0)); tie = int(np.sum(D[2:] == 0))
    print(f"x = 2〜{N} のうち、3 余る方が多い x：{lead3} 個、同数：{tie} 個、1 余る方が多い x：{lead1} 個")
    print(f"初めて 1 余る方が多くなる x：{first}")
    return D

# ---------- 発展：4 で割って 1 余る数だけの世界（H の世界） ----------
def h_numbers(limit):
    return [h for h in range(1, limit + 1) if h % 4 == 1]

def is_h_prime(h, _cache={}):
    """H の世界の「素数」：1 より大きく、H の世界の二つの数（どちらも 1 より大きい）の積に書けない"""
    if h in _cache:
        return _cache[h]
    ok = h > 1 and h % 4 == 1 and not any(h % a == 0 and (h // a) % 4 == 1 for a in range(5, int(h ** 0.5) + 1, 4))
    _cache[h] = ok
    return ok

def h_factorizations(n):
    """H の世界で n を H-素数の積に分けるやり方をすべて（順序は無視）"""
    res = set()
    def rec(m, start, cur):
        if m == 1:
            res.add(tuple(cur)); return
        for q in range(start, int(m ** 0.5) + 1, 4):
            if m % q == 0 and is_h_prime(q):
                rec(m // q, q, cur + [q])
        if m >= start and is_h_prime(m):
            res.add(tuple(cur + [m]))
    rec(n, 5, [])
    return sorted(res)

def h_split(n):
    """H の世界での分け方（因数の木用）：最小の H-因数で分ける"""
    for a in range(5, int(n ** 0.5) + 1, 4):
        if n % a == 0:
            return a, n // a
    return None

# ---------- 発展：素数は無限にある（ユークリッドの数） ----------
def euclid_numbers(k=10):
    rows = []
    prod = 1
    for i, p in enumerate(sympy.primerange(2, 1000), 1):
        if i > k:
            break
        prod *= p
        n = prod + 1
        f = sympy.factorint(n)
        new = [q for q in f if q > p]
        rows.append((i, p, n, factorization_str(n), new))
    print(f"{'k':>2} {'最後の素数 p':>10} {'2×3×…×p + 1':>14}  {_pad('素因数分解', 22)}新しく出た素数")
    for i, p, n, fs, new in rows:
        print(f"{i:>2} {p:>10} {n:>14}  {_pad(fs, 22)}{', '.join(map(str, new))}")
    return rows

## 1. 単元の地図（教科書の地図）

四角は **事実**（ノード）。矢印は「教科書ならこう並べる」という **一案** です。
矢印は正解ではありません。きみの地図では、別の道でつながっていてかまいません。

In [ ]:
draw_reference()

## 2. 自分の地図をつくる

授業で習ったところに「聞いた」の印をつけます。まだ何も確かめていないので、地図は白いままです。

In [ ]:
me = new_map("Aさん")        # ← 自分の名前
heard(me)                     # 授業で聞いたところ全部に「聞いた」の印（発展は「まだ」）
snapshot(me, "授業のあと")     # いまの地図を記録しておく
show_map(me, save="map_0_after_class")

## 3. 実験で確かめる

実験のあとに、自分で印をつけます。書き方は

```python
mark(me, ノード, 印の番号, "わけ（自分の言葉で）", evidence="どの実験か")
link(me, ノードA, ノードB, "どんなつながりか")
```

「わけ」はきみの言葉で。うまく書けないところは、まだ「たしかめた」止まりで大丈夫です。

### 実験1　エラトステネスの篩 — 素数を「見つける」

2 の倍数、3 の倍数、5 の倍数、… と順に消していきます。消されずに残った数が素数です。

In [ ]:
primes = sieve_grid(100)
print(f"100 までの素数は {len(primes)} 個：", primes)

**見るところ**

- 1 は消されなかった。でも 1 は素数？ （素数の約束：約数が「1 とその数」の **2 つ** ある数。1 の約数は 1 つだけ）
- 消された数の右下の小さい数は、その数を消した素数。7 までの素数で消すだけで、100 まで全部きまったのはなぜ？

In [ ]:
mark(me, "prime", 2, "100までの表で、消されずに残った数。約数が1とその数の2つだけ", evidence="実験1")
mark(me, "composite", 2, "消された数。右下に書いてある小さい素数で割り切れる", evidence="実験1")
mark(me, "sieve", 2, "2,3,5,7 の倍数を消すだけで 100 まで全部きまった（なぜ 7 までで足りるかは、まだ）", evidence="実験1")
mark(me, "divisor_multiple", 2, "『倍数を消す』は『その数を約数にもつ数を消す』ことだった", evidence="実験1")
link(me, "divisor_multiple", "sieve", "篩は『倍数を消す』道具")
link(me, "sieve", "prime", "残った数が素数")

### 実験2　割っていく — 素因数分解の手順

小さい素数から順に、割れるだけ割ります。出てきた素数を並べたものが素因数分解です。

In [ ]:
def factorize(n):
    """小さい素数から順に割っていく"""
    steps, p = [], 2
    while n > 1:
        while n % p == 0:
            steps.append(p)
            n //= p
        p += 1
    return steps

for n in [12, 60, 360, 1001, 2024, 65536, 999999]:
    fs = factorize(n)
    print(f"{n:>7} = {' × '.join(map(str, fs)):<32} → {factorization_str(n)}")

# 自分の手順と sympy（数学ソフト）の答えが 2〜10000 で全部一致するか
agree = all(sorted(factorize(n)) == sorted(p for p, e in sympy.factorint(n).items() for _ in range(e))
            for n in range(2, 10001))
print("\n2〜10000 のすべてで sympy と一致：", agree)

**見るところ**

- なぜ必ず終わる？ → 割るたびに数が小さくなるから。
- なぜ出てくるのは素数だけ？ → 1 以外で **いちばん小さい約数** は、必ず素数（もし合成数なら、その約数がもっと小さい約数になってしまう）。

In [ ]:
mark(me, "factorization", 3, "どんな数も素数の積に書けた。わけ：1以外の一番小さい約数は素数。それで割ると数が小さくなるから、いつか素数だけになる", evidence="実験2")
mark(me, "trial_division", 3, "小さい素数から順に割る。割るたびに小さくなるから必ず終わる", evidence="実験2")
mark(me, "exponent", 3, "同じ素数をまとめて 2³×3²×5 と書くだけ。個数を数えれば書ける", evidence="実験2")
link(me, "factorization", "trial_division", "分解のやり方")
link(me, "trial_division", "exponent", "出てきた素数をまとめて書く")

### 実験3　因数の木を、ちがう順番で作る — 分解は「ただ一通り」？

360 を、ちがう分け方で木にしてみます。葉（黒）に出てくる素数は同じになる？

In [ ]:
t1 = factor_tree(360)                                          # 小さい素数から割る
t2 = factor_tree(360, splits={360: (12, 30), 12: (3, 4), 30: (5, 6)})
t3 = factor_tree(360, splits={360: (20, 18), 20: (4, 5), 18: (2, 9)})
show_trees([t1, t2, t3], ["① 小さい素数から", "② 360 = 12 × 30 から", "③ 360 = 20 × 18 から"])

# 2〜2000 のすべての数で、でたらめな順番に分けても葉の組が同じになるか
rng = random.Random(0)
same = all(sorted(leaves(factor_tree(n, rng=rng))) == sorted(leaves(factor_tree(n, rng=rng)))
           for n in range(2, 2001))
print("2〜2000 のすべての数で、どんな順番に分けても葉（素数）の組は同じ：", same)

**見るところ**

- 順番を変えても、葉の組は同じだった（2000 まで）。では 2001 以上でも「必ず」同じと言える？ 
  → 見た範囲では同じ。でも「必ず」と言うには、理由が要る。だからこの印は **「たしかめた」** まで。
- もし 1 を素数に入れたら？ → 6 = 2×3 = 1×2×3 = 1×1×2×3 = … と、分解がいくらでも作れて「ただ一通り」が壊れる。

In [ ]:
mark(me, "uniqueness", 2, "360 を3通りに分けても葉は同じ。2〜2000 も全部同じだった。でも『2001以上でも必ず』とは、まだ言えない", evidence="実験3")
mark(me, "one_not_prime", 3, "1 を素数に入れると 6 = 2×3 = 1×2×3 = … と分解がいくらでも作れて『ただ一通り』が壊れる。だから 1 は入れない約束", evidence="実験3")
link(me, "one_not_prime", "uniqueness", "1 を外すから『ただ一通り』と言える")
link(me, "factorization", "uniqueness", "分解はできる。しかも一通り（たぶん）")
snapshot(me, "実験1〜3のあと")
show_map(me, compare=0, save="map_1_after_exp3")

### 実験4（発展）「ただ一通り」は当たり前？ — 4 で割って 1 余る数だけの世界

1, 5, 9, 13, 17, 21, … （4 で割って 1 余る数）だけを使う「H の世界」を考えます。
この世界の数どうしをかけても、世界の外には出ません（5 × 9 = 45 も 4 で割って 1 余る）。
この世界の「素数」は、世界の中の 2 つの数（どちらも 1 より大きい）の積に書けない数、と約束します。

In [ ]:
H = h_numbers(100)
print("H の世界の数（100 まで）  ：", H)
print("H の世界の『素数』（100 まで）：", [h for h in H if is_h_prime(h)])
print("\nH の世界で、分け方が 2 通り以上ある数（1500 まで）：")
for n in h_numbers(1500):
    fs = h_factorizations(n)
    if len(fs) > 1:
        print(f"  {n} = " + " = ".join(" × ".join(map(str, f)) for f in fs))

ta = factor_tree(441, is_prime=is_h_prime, splits={441: (21, 21)})
tb = factor_tree(441, is_prime=is_h_prime, splits={441: (9, 49)})
show_trees([ta, tb], ["H の世界：441 = 21 × 21", "H の世界：441 = 9 × 49"], save="h_trees")

**見るところ**

同じ 441 なのに、葉がちがう。H の世界では「ただ一通り」は成り立ちません。
ふつうの整数の世界で「ただ一通り」が成り立つのは、当たり前ではなく、整数の世界の **特別な性質** です。
（なぜ成り立つかの証明は、高校・大学で出会います。）

In [ ]:
# 印は「たしかめた」のまま。でも「わけ」が深くなった
mark(me, "uniqueness", 2, "H の世界では 441 = 21×21 = 9×49 と 2 通りに分かれる。だから『ただ一通り』は当たり前ではなく、ふつうの整数の特別な性質。なぜ成り立つかは、まだ説明できない", evidence="実験4")

### 実験5　約数の個数 — 規則を見つけて、説明する

素因数分解と、約数の個数を並べてみます。規則が見えるでしょうか。

In [ ]:
def divisors(n):
    return [d for d in range(1, n + 1) if n % d == 0]

print(f"{'n':>4}  {_pad('素因数分解', 14)}{_pad('約数', 46)}個数")
for n in [6, 8, 12, 16, 36, 72, 100, 360]:
    ds = divisors(n)
    print(f"{n:>4}  {_pad(factorization_str(n), 14)}{_pad(str(ds), 46)}{len(ds)}")

# 予想：（指数＋1）を全部かける
def guess(n):
    return math.prod(e + 1 for e in sympy.factorint(n).values())

ok = all(guess(n) == sympy.divisor_count(n) for n in range(1, 10001))
print("\n予想『（指数＋1）をかける』が 1〜10000 で全部あっている：", ok)

**説明してみる**：72 = 2³ × 3² の約数は、「2 を何個使うか（0〜3 個の 4 通り）」と「3 を何個使うか（0〜2 個の 3 通り）」で決まる。

In [ ]:
divisor_grid(72)

表のマスが全部で 4 × 3 = 12 個。これが約数の個数です。

ただし、ここで「表にない約数はない」と言うとき、**約数を分解すると、それは 72 の分解の一部になる（分解はただ一通りだから）** を使っています。
その「ただ一通り」は、まだ「たしかめた」止まりでした。印をつけると、それが見えます。

In [ ]:
mark(me, "divisor_count", 3, "約数は『2 を何個、3 を何個使うか』で決まる（表の行×列）。だから（指数＋1）のかけ算。ただし『表にない約数はない』と言うところで、分解がただ一通りであることを使っている",
     evidence="実験5", uses=["factorization", "uniqueness"])
link(me, "uniqueness", "divisor_count", "『表にない約数はない』と言うのに使う")
check(me)

### 実験6　平方数 — 指数を見る

In [ ]:
for n in [4, 9, 36, 100, 144, 225, 900, 1296]:
    r = math.isqrt(n)
    print(f"{n:>5} = {factorization_str(n):<12} = ({factorization_str(r)})²")
print("平方数でない例：", "、".join(f"{n} = {factorization_str(n)}" for n in [8, 12, 18, 50, 72]))

mark(me, "square", 3, "指数がすべて偶数なら、半分ずつに分けて (…)² と書ける。だから平方数。逆（平方数なら指数が偶数）は、分解がただ一通りだから、と思うけれど自信がない",
     evidence="実験6", uses=["exponent", "factorization"])
link(me, "exponent", "square", "指数が全部偶数なら平方数")

### 実験7（発展）素数は無限にある？ — 全部かけて 1 を足す

小さい方から k 個の素数を全部かけて、1 を足します。その数はどの素数で割っても 1 余るはず。素因数分解するとどうなる？

In [ ]:
rows = euclid_numbers(10)

**見るところ**：素数になることもあれば（3, 7, 31, …）、ならないこともある（30031 = 59 × 509）。でも、出てくる素因数はいつも **リストにない新しい素数**。

**証明（言葉で）**：素数が有限個だったとして、全部かけて 1 を足した数を N とする。N はどの素数で割っても 1 余る。
でも N には素因数が必ずある（実験2）。その素因数はリストにない新しい素数。矛盾。だから素数は無限にある。

In [ ]:
mark(me, "infinite_primes", 4, "有限個だとして全部かけて 1 を足すと、どの素数でも割り切れない。でも素因数は必ずある。だからリストにない素数がある。いつでも成り立つ理由を言えた",
     evidence="実験7", uses=["factorization"])

警告が出ました。この証明は「どんな数にも素因数がある」を使っています。それが「説明できる」止まりなら、証明も「説明できる」止まりです。
そこで、実験2 の説明をきちんと書き直して「証明できる」まで上げます。

In [ ]:
mark(me, "factorization", 4, "1 より大きい数 n の、1 以外で一番小さい約数 p は素数（p が合成数なら p の約数 q は n のもっと小さい約数になって矛盾）。n を p で割って繰り返せば有限回で終わる",
     evidence="実験7")
link(me, "factorization", "infinite_primes", "証明に『素因数が必ずある』を使う")
check(me)

### 実験8（発展）素数のレース — 「たしかめた」と「証明できる」はちがう

2 以外の素数を 4 で割ると、余りは 1 か 3。どちらの素数が多いでしょう？ 数えてみます。

In [ ]:
D = prime_race(30000)

**見るところ**

- 26860 まで数えても、ずっと「3 余る素数」の方が多い。ここで「いつも 3 余る方が多い」と言いたくなる。
- でも 26861 で初めて逆転し、すぐ戻る。ずっと先（およそ 62 万）でも、また逆転する。
- 「何回でも逆転する」ことは 1914 年に証明されている（Littlewood）。でも、その証明は中学の道具では書けない。
  だから、この印は **「たしかめた」止まり**。それでいい。3 万まで見たことと、いつでも成り立つことは、別のことだから。

In [ ]:
mark(me, "prime_race", 2, "30000 まで数えた。26860 までは 3 余る方がずっと多い。26861 で初めて 1 余る方が多くなり、すぐ戻る。『いつも』も『何回も逆転』も、自分では証明できない",
     evidence="実験8")
link(me, "prime", "prime_race", "素数を 4 で割った余りで分ける")

## 4. 地図の成長を見る

実験1〜3 のあとの記録と比べて、どこが変わったかを二重枠で示します。

In [ ]:
snapshot(me, "すべての実験のあと")
show_map(me, compare=1, save="map_2_final_diff")
show_map(me, title="Aさんの地図（すべての実験のあと）", save="map_2_final")

In [ ]:
diff(me, 0, 2)
show_history(me)      # 名前なしの縮小版。位置は上の地図と同じ

**地図に残ったもの**

- 黒（証明できる）：素数は無限にある、素因数分解ができること
- 濃い灰色（説明できる）：1 は素数ではない、手順、累乗、約数の個数、平方数
- 薄い灰色（たしかめた）：**分解はただ一通り**、**素数の偏り**、篩、素数、合成数
- 白（聞いた）：最大公約数・最小公倍数 — まだ実験していない

「約数の個数」から「一意性」へ伸びる **黒の点線** は、「一意性が正しいとすれば、説明できる」という印。
証明の途中で、何を借りているかが地図に見えます。

## 5. 問いかけ — 地図は作らない、聞くだけ

先生（や AI）の役目は、地図を代わりに作ることではなく、地図を見て **問う** ことです。
問いは、地図の状態から決まります：「聞いた」のままのノード、「たしかめた」止まりのノード、弱い材料で説明しているところ、わけのない矢印。

In [ ]:
qs = ask_me(me, k=7)

上の問いは、決まった規則で作っています。生成 AI を使う場合も、役目は同じにします：**地図を作り直さない。答えを言わない。根拠を尋ねる。** 下はそのための指示文の例です。

In [ ]:
LLM_PROMPT = f"""あなたは数学の先生です。次は生徒の「自分の地図」（JSON）です。
ルール：地図を作り直さない。答えを言わない。問いは 3 つまで。
問いは「根拠」を尋ねるものに限る（どこまで確かめた？ なぜそう言える？ 何を前提にしている？）。

生徒の地図：
{json.dumps({name_of(k): {"印": LEVELS[v["level"]], "わけ": v["why"], "使った知識": [name_of(u) for u in v["uses"]]}
             for k, v in me["nodes"].items()}, ensure_ascii=False, indent=1)}
"""
print(LLM_PROMPT[:700] + "\n…")
# 実際に生成 AI に渡すときは、ここに API 呼び出しを書く（このプロトタイプでは呼ばない）

## 6. 保存する

自分の地図は JSON ファイルに保存できます（JupyterLite ではブラウザの中に保存されます）。
次の授業では読み込んで、続きから印をつけます。

In [ ]:
save_map(me, "my_map.json")
me2 = load_map("my_map.json")
print(me2["name"], "／", len(me2["snapshots"]), "回の記録 ／", len(me2["log"]), "件のログ")

---
## 先生・研究者向けの設計メモ

**ねらい**：教科書の知識グラフ（ノード＝事実、矢印＝並べ方の一案）を土台に、生徒一人ひとりが「自分が納得した知識」を自分の地図として持ち、実験（Jupyter Notebook）でノードを **接地** させていく。地図は理解の到達点ではなく、**納得の来歴** を表す。

**設計の原則と、このノートでの実装**

| 原則 | このノートでは |
|:--|:--|
| (a) 根拠・接地状態をノードに持たせる | `mark(node, level, why, evidence, uses)`：印（0〜4）、わけ、根拠となる実験、説明に使った知識 |
| (b) 個人の地図は変化する。差分が成長の実感 | `snapshot` / `diff` / `show_map(compare=)` / `show_history`：二重枠と ↑ で変化を示す |
| (c) 事実は定まるが、矢印は選択 | 教科書の矢印は薄い灰色の背景。生徒の矢印（`link`）は黒。一致度は採点しない |
| (d) 実験による納得と証明の区別 | 「たしかめた（2）」と「証明できる（4）」を分ける。一意性・素数の偏りは 2 のまま残す。素数レースは 26861 で逆転 |
| (d') 説明が何を借りているかを見せる | `uses` と `check()`：弱い材料を使った説明は ⚠ と黒の点線で表示（「〜が正しいとすれば」） |
| (e) 道具を軽く、小さく始める | 14 ノード、1 ファイル、依存は sympy / networkx / matplotlib のみ。JupyterLite（Pyodide）で動く。紙から始めてもよい |
| (f) AI は作る側でなく問う側 | `ask_me()` は規則で問いを作る。LLM 用の指示文も「地図を作り直さない・答えを言わない・根拠を尋ねる」 |
| (g) グラフ由来の指標を目的化しない | ノード数・接地率の集計や採点機能は意図的に付けていない。評価は転移課題・説明課題・動機づけ尺度で行う |

**評価で見るもの／見ないもの**：見る — 転移課題（新しい数の分解・一意性を使う応用）、説明課題（「なぜ 1 は素数でないか」）、正当化の型（Harel–Sowder の証明スキーム：外的／経験的／演繹的）、内発的動機づけ（IMI など）。見ない — ノード数、印の平均。比較条件は「実験はするが地図は作らない」群を置く。

**接地レベルの補足**：レベルは序列ではなく種類の違いを含む。経験的な確信（2）は専門家も使う正当な段階で、そこで止まることを隠さないのがこのノートの誠実さ。de Villiers の言う通り、確信はしばしば証明に先立つ。

**動かし方**：ローカルの Jupyter（Python 3、`pip install sympy networkx matplotlib`）。JupyterLite では `files/` にこのノートと `fonts/` を置く。図の日本語表示のフォントは、同梱の `fonts/`（Noto Sans CJK JP のサブセット）→ パソコンの日本語フォント → インターネット の順に `setup_font()` が探す。SageMath カーネルでもそのまま動く。

**主な参考**：Novak & Gowin (1984) *Learning How to Learn*；Nesbit & Adesope (2006) *RER* 76(3)；Schroeder et al. (2018) *Educ. Psychol. Rev.* 30（作成 g=0.72 ＞ 閲覧 g=0.43）；Bull & Kay (2013) Open Learner Models；Harel & Sowder (1998) proof schemes；de Villiers (1990) *Pythagoras* 24；Leech (1957) *J. London Math. Soc.*（26861）；Rubinstein & Sarnak (1994) *Exp. Math.* 3(3)；Littlewood (1914)。